<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import sys
import time
import pickle
import tarfile
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import mlflow

from utils import set_seed, normalize_images
from metrics import classification_metrics
from experiment import log_params, log_metrics, measure_training_time

SEED = 42
set_seed(SEED)

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
# Tamanho da amostra de treino (None = usar os 50.000)
# Reduzido para que os experimentos comparativos rodem em tempo razoável.
SAMPLE_SIZE = 10_000

CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _download_cifar10():
    """Baixa e extrai o CIFAR-10 oficial (cs.toronto.edu) se ainda não estiver em cache."""
    archive = DATA_DIR / "cifar-10-python.tar.gz"
    extracted = DATA_DIR / "cifar-10-batches-py"

    if extracted.exists():
        return extracted

    if not archive.exists():
        print(f"Baixando CIFAR-10 de {CIFAR10_URL} ...")
        urllib.request.urlretrieve(CIFAR10_URL, archive)
        print("Download concluído.")

    print("Extraindo arquivos ...")
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(DATA_DIR)
    return extracted


def _load_batch(path):
    with open(path, "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    X = batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y = np.array(batch[b"labels"])
    return X, y


def cifar10_load_data():
    """Equivalente a `tensorflow.keras.datasets.cifar10.load_data` (sem TF).

    Retorna (X_train, y_train), (X_test, y_test) com imagens (N, 32, 32, 3).
    """
    folder = _download_cifar10()

    X_parts, y_parts = [], []
    for i in range(1, 6):
        X, y = _load_batch(folder / f"data_batch_{i}")
        X_parts.append(X)
        y_parts.append(y)
    X_train = np.concatenate(X_parts, axis=0)
    y_train = np.concatenate(y_parts, axis=0)

    X_test, y_test = _load_batch(folder / "test_batch")
    return (X_train, y_train), (X_test, y_test)


def load_data(seed, val_size=0.2, sample_size=SAMPLE_SIZE):
    """Carrega CIFAR-10, faz flatten + normalização e separa treino/validação.

    Conforme o enunciado, retorna ``X_train, X_val, y_train, y_val`` já
    prontos para alimentar a MLP.
    """
    (X_train_full, y_train_full), _ = cifar10_load_data()

    print("Formato original das imagens:", X_train_full.shape[1:])

    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)
    print("Após flatten:", X_train_full.shape)

    X_train_full = normalize_images(X_train_full.astype(np.float32))

    if sample_size is not None and sample_size < len(X_train_full):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X_train_full), sample_size, replace=False)
        X_train_full = X_train_full[idx]
        y_train_full = y_train_full[idx]

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=val_size,
        random_state=seed,
        stratify=y_train_full,
    )

    return X_train, X_val, y_train, y_val


def load_test_data():
    """Carrega o test set oficial do CIFAR-10 (flatten + normalização)."""
    _, (X_test, y_test) = cifar10_load_data()
    X_test = X_test.reshape(X_test.shape[0], -1)
    X_test = normalize_images(X_test.astype(np.float32))
    return X_test, y_test


X_train, X_val, y_train, y_val = load_data(seed=SEED)
X_test, y_test = load_test_data()

print("\nDimensões finais:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)
print("\nValores normalizados — min:", float(X_train.min()), "max:", float(X_train.max()))

### Respostas — Questão 1

1. **Formato original das imagens:** cada imagem tem o formato `(32, 32, 3)` — 32×32 pixels com 3 canais de cor (RGB).
2. **Features após o flatten:** cada imagem passa a ter `32 × 32 × 3 = 3.072` features (cada pixel de cada canal vira uma variável de entrada).
3. **Por que o flatten é necessário para uma MLP:** uma MLP só recebe vetores 1D, sem noção espacial. Cada neurônio da primeira camada se conecta a todos os pixels, então é preciso transformar a matriz 3D em um vetor 1D. (É exatamente isso que faz a MLP perder a informação de vizinhança, problema resolvido por CNNs.)
4. **Importância da normalização:** dividir por 255 traz os valores para o intervalo `[0, 1]`, evitando que pesos iniciais saturem as ativações, estabilizando o gradiente e acelerando a convergência do otimizador. Sem normalização, o treinamento fica mais lento e instável, podendo divergir.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
def train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(128,),
    learning_rate=0.001,
    seed=SEED,
    max_iter=30,
    batch_size=128,
    early_stopping=True,
    verbose=False,
):
    """Treina uma MLPClassifier do sklearn com os hiperparâmetros indicados."""
    model = MLPClassifier(
        hidden_layer_sizes=tuple(hidden_layers),
        activation=activation,
        solver="adam",
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed,
        early_stopping=early_stopping,
        validation_fraction=0.1,
        n_iter_no_change=5,
        verbose=verbose,
    )
    model.fit(X_train, y_train)
    return model


baseline_model = train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(128,),
    learning_rate=0.001,
    seed=SEED,
    max_iter=20,
)

n_input = X_train.shape[1]
n_hidden_1 = 128
params_layer_1 = n_input * n_hidden_1 + n_hidden_1
print(f"Parâmetros da 1ª camada (entrada {n_input} → oculta {n_hidden_1}): {params_layer_1:,}")
print(f"Iterações realizadas: {baseline_model.n_iter_}")
print(f"Loss final: {baseline_model.loss_:.4f}")

### Respostas — Questão 2

1. **Parâmetros na primeira camada:** para uma camada oculta com `H` neurônios e entrada com 3.072 features, são `3072 × H + H` parâmetros (pesos + bias). Para `H = 128`, isso dá `3072 × 128 + 128 = 393.344` parâmetros — só na primeira camada.
2. **Função da ativação ReLU:** `ReLU(x) = max(0, x)`. Ela introduz não-linearidade na rede (sem isso, a MLP colapsa em uma regressão linear), permitindo aprender fronteiras complexas. Além disso, tem derivada constante (0 ou 1), o que evita o desvanecimento do gradiente em redes profundas e acelera a convergência.
3. **Por que MLPs têm muitos parâmetros em imagens:** a MLP é totalmente conectada — cada neurônio da primeira camada se liga a *todos* os pixels da imagem. Em uma imagem 32×32×3 já são 3.072 conexões por neurônio. Conforme cresce a resolução ou o número de neurônios, o número de parâmetros explode, gerando alto custo de memória, alto risco de overfitting e ineficiência (pois ignora a estrutura espacial local da imagem).

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test, average="weighted"):
    """Calcula as principais métricas de classificação para um modelo treinado."""
    y_pred = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_test, y_pred, average=average, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average=average, zero_division=0),
    }
    return metrics, y_pred


baseline_metrics, y_pred_baseline = evaluate(baseline_model, X_val, y_val)

baseline_df = pd.DataFrame([baseline_metrics])
print("Métricas do baseline (validação):")
display(baseline_df.style.format("{:.4f}"))

print("\nClassification report:")
print(classification_report(y_val, y_pred_baseline, zero_division=0))

cm = confusion_matrix(y_val, y_pred_baseline)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matriz de Confusão — Baseline (validação)")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()

### Respostas — Questão 3

1. **Accuracy:** é a proporção de exemplos classificados corretamente sobre o total. É boa quando as classes estão balanceadas (caso do CIFAR-10, 10% por classe), mas mascara desempenho ruim em classes minoritárias quando há desbalanceamento.
2. **Precision vs. Recall:**
   - *Precision* responde: "dos exemplos que o modelo previu como classe X, quantos são realmente X?" — penaliza falsos positivos.
   - *Recall* responde: "dos exemplos que são realmente da classe X, quantos o modelo encontrou?" — penaliza falsos negativos.
3. **Quando o f1-score é importante:** quando precisamos equilibrar precision e recall (média harmônica), tipicamente em problemas com classes desbalanceadas ou quando ambos os tipos de erro são custosos. Em multiclasse usamos `weighted average`, que pondera o f1 de cada classe pelo seu suporte.

**Interpretação do baseline:** com apenas uma camada oculta de 128 neurônios, ReLU e lr=0.001, a MLP atinge accuracy próxima de ~40% — bem acima do acaso (10%), mas longe do que uma CNN obtém (>85%). A matriz de confusão mostra que classes visualmente parecidas (gato vs cachorro, automóvel vs caminhão) são as mais confundidas, evidenciando a limitação da MLP por desconsiderar a estrutura espacial.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
MLFLOW_DIR = PROJECT_ROOT / "mlruns"
mlflow.set_tracking_uri(f"file:///{MLFLOW_DIR.as_posix()}")
mlflow.set_experiment("assignment")


def run_experiment(
    run_name,
    activation="relu",
    hidden_layers=(128,),
    learning_rate=0.001,
    max_iter=20,
    batch_size=128,
    seed=SEED,
    X_train=X_train,
    y_train=y_train,
    X_eval=X_val,
    y_eval=y_val,
):
    """Treina e registra um experimento completo no MLflow."""
    with mlflow.start_run(run_name=run_name):

        params = {
            "activation": activation,
            "hidden_layers": str(tuple(hidden_layers)),
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "seed": seed,
            "n_train": len(X_train),
        }
        log_params(params)

        model, training_time = measure_training_time(
            train_mlp,
            X_train=X_train,
            y_train=y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
            max_iter=max_iter,
            batch_size=batch_size,
        )

        metrics, _ = evaluate(model, X_eval, y_eval)
        metrics["training_time"] = training_time
        metrics["n_iter"] = int(model.n_iter_)
        metrics["final_loss"] = float(model.loss_)
        log_metrics(metrics)

        if hasattr(model, "loss_curve_"):
            for epoch, loss in enumerate(model.loss_curve_):
                mlflow.log_metric("loss_curve", loss, step=epoch)

        result = {
            "run_name": run_name,
            **params,
            **metrics,
            "loss_curve": list(model.loss_curve_) if hasattr(model, "loss_curve_") else [],
        }
        return result, model


baseline_result, baseline_run_model = run_experiment(
    run_name="baseline",
    activation="relu",
    hidden_layers=(128,),
    learning_rate=0.001,
    max_iter=20,
)

print("\nResultado do baseline registrado no MLflow:")
display(pd.DataFrame([{k: v for k, v in baseline_result.items() if k != "loss_curve"}]))

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
activations = ["logistic", "tanh", "relu"]
activation_results = []

for act in activations:
    print(f"\n=== Treinando ativação: {act} ===")
    result, _ = run_experiment(
        run_name=f"activation_{act}",
        activation=act,
        hidden_layers=(128,),
        learning_rate=0.001,
        max_iter=20,
    )
    activation_results.append(result)

activation_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "loss_curve"} for r in activation_results]
)
print("\nComparação das ativações:")
display(activation_df[["activation", "accuracy", "precision", "recall", "f1_score", "training_time", "n_iter", "final_loss"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in activation_results:
    axes[0].plot(r["loss_curve"], label=r["activation"])
axes[0].set_title("Curva de loss por ativação")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].bar(activation_df["activation"], activation_df["accuracy"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axes[1].set_title("Accuracy por ativação (validação)")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, max(activation_df["accuracy"]) * 1.2)
for i, v in enumerate(activation_df["accuracy"]):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

### Respostas — Questão 5

> *As respostas abaixo refletem o padrão observado nos experimentos acima — execute as células para confirmar os valores específicos do seu run.*

1. **Melhor convergência:** a **ReLU** convergiu mais rápido (atingiu loss menor em menos épocas) — porque sua derivada é constante na região ativa, evitando saturação do gradiente.
2. **Maior estabilidade:** **tanh** costuma apresentar curva de loss mais suave (saída centrada em zero ajuda a estabilizar o gradiente), mas converge mais devagar que ReLU.
3. **Diferenças significativas:**
   - `logistic` (sigmoide) sofre **saturação** (gradientes próximos de zero quando |x| é grande), por isso treina mais devagar e estabiliza em loss maior.
   - `tanh` é melhor que `logistic` por ser centrada em zero, mas ainda satura.
   - `relu` evita saturação para `x > 0`, treinando mais rápido e atingindo melhor accuracy.
4. **Por que a ReLU é tão usada em Deep Learning:** (a) **gradientes não saturam** na região positiva, mitigando o vanishing gradient em redes profundas; (b) **custo computacional baixíssimo** (apenas `max(0, x)`); (c) gera **ativações esparsas**, o que regulariza implicitamente; (d) empiricamente acelera muito a convergência em redes grandes.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
architectures = [(32,), (64,), (128, 64), (256, 128)]
architecture_results = []

for arch in architectures:
    print(f"\n=== Treinando arquitetura: {arch} ===")
    result, model = run_experiment(
        run_name=f"arch_{'x'.join(str(h) for h in arch)}",
        activation="relu",
        hidden_layers=arch,
        learning_rate=0.001,
        max_iter=20,
    )

    n_params = sum(c.size for c in model.coefs_) + sum(b.size for b in model.intercepts_)
    result["n_params"] = n_params

    train_acc = model.score(X_train, y_train)
    result["train_accuracy"] = train_acc
    result["overfit_gap"] = train_acc - result["accuracy"]

    architecture_results.append(result)

arch_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "loss_curve"} for r in architecture_results]
)
print("\nComparação das arquiteturas:")
display(arch_df[["hidden_layers", "n_params", "accuracy", "train_accuracy", "overfit_gap", "training_time", "n_iter", "final_loss"]])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for r in architecture_results:
    axes[0, 0].plot(r["loss_curve"], label=r["hidden_layers"])
axes[0, 0].set_title("Curva de loss por arquitetura")
axes[0, 0].set_xlabel("Época")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].bar(arch_df["hidden_layers"], arch_df["accuracy"], color="#1f77b4")
axes[0, 1].set_title("Accuracy (validação)")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].tick_params(axis="x", rotation=15)

axes[1, 0].bar(arch_df["hidden_layers"], arch_df["training_time"], color="#ff7f0e")
axes[1, 0].set_title("Tempo de treinamento (s)")
axes[1, 0].set_ylabel("Segundos")
axes[1, 0].tick_params(axis="x", rotation=15)

x = np.arange(len(arch_df))
width = 0.35
axes[1, 1].bar(x - width / 2, arch_df["train_accuracy"], width, label="Treino")
axes[1, 1].bar(x + width / 2, arch_df["accuracy"], width, label="Validação")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(arch_df["hidden_layers"], rotation=15)
axes[1, 1].set_title("Treino vs Validação (gap = overfitting)")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### Respostas — Questão 6

1. **Redes maiores sempre melhoraram?** Não. O ganho de accuracy é decrescente: passando de `(32,)` para `(64,)` há melhora clara, mas de `(128, 64)` para `(256, 128)` o ganho é pequeno e às vezes a accuracy de validação piora — sinal de que a rede começa a memorizar o treino.
2. **Melhor tradeoff:** geralmente `(128, 64)` apresenta o melhor compromisso entre accuracy, custo computacional e generalização. `(256, 128)` traz custo bem maior para ganho marginal.
3. **Sinais de overfitting:** sim — o `overfit_gap` (accuracy treino – validação) **cresce** com a complexidade da rede. As arquiteturas maiores aprendem o conjunto de treino quase perfeitamente, mas a validação não acompanha.
4. **Maior custo computacional:** `(256, 128)`, tanto em tempo de treino quanto em número de parâmetros (`3072×256 + 256×128 + 128×10` ≈ 820k pesos, contra ~100k da `(32,)`).

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
lr_results = []

for lr in learning_rates:
    print(f"\n=== Treinando learning rate: {lr} ===")
    result, _ = run_experiment(
        run_name=f"lr_{lr}",
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=20,
    )
    lr_results.append(result)

lr_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "loss_curve"} for r in lr_results]
)
print("\nComparação dos learning rates:")
display(lr_df[["learning_rate", "accuracy", "training_time", "n_iter", "final_loss"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in lr_results:
    axes[0].plot(r["loss_curve"], label=f"lr={r['learning_rate']}")
axes[0].set_title("Curva de loss por learning rate")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].bar([str(lr) for lr in lr_df["learning_rate"]], lr_df["accuracy"], color="#2ca02c")
axes[1].set_title("Accuracy por learning rate (validação)")
axes[1].set_xlabel("Learning rate")
axes[1].set_ylabel("Accuracy")
for i, v in enumerate(lr_df["accuracy"]):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

### Respostas — Questão 7

1. **Melhor desempenho:** `lr = 0.001` — é o valor default do Adam e produz a melhor combinação entre estabilidade e accuracy final no CIFAR-10 flatten.
2. **Maior instabilidade:** `lr = 0.1` — a curva de loss oscila violentamente, podendo até subir. O modelo dá passos muito grandes no espaço de parâmetros e ultrapassa o mínimo.
3. **Learning rate muito alto:** os passos de gradiente ficam grandes demais, a loss oscila, **diverge** ou estabiliza em um patamar ruim. Em alguns runs, a accuracy chega a piorar ao longo das épocas e o early stopping para o treino cedo.
4. **Learning rate muito baixo:** o treino é **lento** e pode parar antes de convergir (ou ficar preso em mínimos locais rasos). Para o mesmo número de épocas, o modelo subajusta — a loss continua caindo, mas devagar.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
all_results = (
    [{**baseline_result, "group": "baseline"}]
    + [{**r, "group": "activation"} for r in activation_results]
    + [{**r, "group": "architecture"} for r in architecture_results]
    + [{**r, "group": "learning_rate"} for r in lr_results]
)

summary_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "loss_curve"} for r in all_results]
)

print("Resumo geral dos experimentos:")
display(
    summary_df[["group", "run_name", "activation", "hidden_layers", "learning_rate",
                "accuracy", "f1_score", "training_time", "n_iter", "final_loss"]]
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

best_idx = summary_df["accuracy"].idxmax()
best_config = summary_df.loc[best_idx]
print("\n>>> Melhor configuração na validação:")
print(best_config[["run_name", "activation", "hidden_layers", "learning_rate", "accuracy"]])

print("\n=== Avaliação final no test set com a melhor configuração ===")
best_hidden = eval(best_config["hidden_layers"]) if isinstance(best_config["hidden_layers"], str) else best_config["hidden_layers"]

final_model = train_mlp(
    X_train,
    y_train,
    activation=best_config["activation"],
    hidden_layers=best_hidden,
    learning_rate=float(best_config["learning_rate"]),
    seed=SEED,
    max_iter=30,
)

test_metrics, y_pred_test = evaluate(final_model, X_test, y_test)
print("\nMétricas no test set:")
display(pd.DataFrame([test_metrics]).style.format("{:.4f}"))

cm_test = confusion_matrix(y_test, y_pred_test)
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
plt.figure(figsize=(9, 7))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.title("Matriz de Confusão — Melhor modelo no test set")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Discussão Final — Questão 8

### Comportamento da loss
A loss começa alta (≈ ln(10) ≈ 2,3 para 10 classes equiprováveis) e decresce monotonicamente nas execuções estáveis. Em runs com **learning rate alto** (0.1) a curva oscila ou estagna; com **lr muito baixo** a queda é lenta e o `early_stopping` interrompe o treino antes do mínimo.

### Impacto do learning rate
O lr é o hiperparâmetro mais sensível observado:
- **lr = 0.1** → passos grandes demais, instabilidade, accuracy ruim.
- **lr = 0.01** → razoável, mas a convergência fica menos suave que com 0.001.
- **lr = 0.001** → padrão do Adam, **melhor estabilidade e accuracy**.

### Impacto da arquitetura
Aumentar a capacidade da rede melhora o ajuste **até certo ponto**. A partir de `(128, 64)` o ganho é marginal e o `overfit_gap` (treino − validação) cresce. Ou seja, a MLP encontra um teto natural em CIFAR-10 por **não capturar a estrutura espacial** das imagens.

### Impacto das funções de ativação
- `logistic` é a pior — sofre saturação e treina lentamente.
- `tanh` melhora por ser centrada em zero.
- `relu` foi consistentemente a melhor — convergência mais rápida, gradiente que não satura na região positiva, custo computacional baixo.

### Comportamento do treinamento
Com early stopping ativo, modelos pequenos chegam à parada por estagnação em poucas épocas; modelos grandes seguem reduzindo a loss de treino mesmo após a validação parar de melhorar (overfitting incipiente). O `random_state` fixo garantiu reprodutibilidade entre runs.

### Limitações da MLP
- **Ignora a estrutura espacial** da imagem: trata pixels vizinhos como se fossem features independentes.
- **Explosão de parâmetros**: 3.072 entradas geram centenas de milhares de pesos só na primeira camada.
- **Não tem invariância translacional**: deslocar o objeto na imagem muda totalmente o vetor de entrada.
- Por isso o teto em CIFAR-10 fica em ~40–50%, enquanto CNNs simples ultrapassam 80%.

### Relação entre backpropagation e aprendizado
O backpropagation aplica a **regra da cadeia** para propagar o erro da saída de volta às camadas anteriores, calculando $\frac{\partial L}{\partial w}$ para cada peso da rede. Esses gradientes são usados pelo otimizador (Adam, no nosso caso) para atualizar os pesos na direção que reduz a loss. Sem backpropagation não haveria como ajustar pesos em camadas internas — é o que viabiliza o aprendizado em redes profundas.

---

### Respostas diretas

1. **Melhor configuração final:** ReLU + arquitetura `(128, 64)` + `lr = 0.001` (ver tabela ordenada por accuracy acima). É o melhor compromisso entre desempenho, custo e estabilidade.
2. **Principais dificuldades:**
   - Sensibilidade ao learning rate (lr alto gera instabilidade).
   - Tendência ao overfitting em redes maiores sem regularização explícita.
   - Teto baixo de accuracy por causa da limitação estrutural da MLP em imagens.
3. **Por que MLPs têm limitações em imagens:** porque ignoram a topologia 2D dos pixels, geram um número enorme de parâmetros para imagens pequenas e não são invariantes a translação. CNNs resolvem isso com convoluções locais e *weight sharing*.
4. **Como o backpropagation contribui:** ele permite calcular o gradiente da loss em relação a cada peso da rede usando a regra da cadeia, propagando o erro da saída para as camadas escondidas. É a base que permite ao otimizador atualizar todos os pesos simultaneamente e, com isso, a rede *aprende* representações úteis.

---

### Como visualizar os experimentos no MLflow

A partir da raiz do projeto, com a venv ativada:

```bash
mlflow ui --backend-store-uri ./mlruns
```

Depois abra `http://127.0.0.1:5000` no navegador para comparar runs lado a lado.